In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
import torch 
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
WHEELHOUSE="/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128"

!python - <<'PY'
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA torch:", torch.version.cuda)
!find /kaggle/input -maxdepth 2 -type f | sed -n '1,250p'

In [ ]:
%%bash
set -euo pipefail

WHEELHOUSE="/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128"

echo "WHEELHOUSE=$WHEELHOUSE"
test -f "$WHEELHOUSE/huggingface_hub-0.36.0-py3-none-any.whl"
test -f "$WHEELHOUSE/json_repair-0.63.4-py3-none-any.whl"

find "$WHEELHOUSE" -maxdepth 1 -type f -name '*.whl' ! -name '*cp313*' -print0 \
  | sort -z \
  | xargs -0 -n 1 python -m pip install --no-index --no-deps --force-reinstall

In [ ]:
%%bash
set -euo pipefail

WHEELHOUSE="/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128"

python -m pip install \
  --no-index \
  --find-links "$WHEELHOUSE" \
  --no-deps \
  --ignore-installed \
  --force-reinstall \
  numpy==2.2.6

In [ ]:
import sys

WHEELHOUSE = "/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128"

!{sys.executable} -m pip install --no-index --no-deps --force-reinstall \
  "$WHEELHOUSE/modelscope-1.31.0-py3-none-any.whl" \
  "$WHEELHOUSE/json_repair-0.63.4-py3-none-any.whl"

In [ ]:
%%bash
set -euo pipefail

WHEELHOUSE="/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128"

python -m pip install \
  --no-index \
  --find-links "$WHEELHOUSE" \
  --no-deps \
  --ignore-installed \
  --force-reinstall \
  pyarrow==20.0.0


In [ ]:
import deepspeed

In [ ]:
import peft, datasets, pyarrow, modelscope, json_repair

print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("pyarrow:", pyarrow.__version__)
print("modelscope:", modelscope.__version__)
print("json-repair:", json_repair.__version__)

In [ ]:
import torch, swift, transformers, trl, peft, qwen_vl_utils, decord
print("torch:", torch.__version__)
print("swift:", swift.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)


In [ ]:
cp -a /kaggle/input/datasets/vucongaaa/dar-r1/DAR /kaggle/working/DAR


In [ ]:
from pathlib import Path
import json
import os

source = Path(
    "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2"
)
target = Path("/kaggle/working/qwen2_5_vl_3b_fixed")
target.mkdir(parents=True, exist_ok=True)

# Symlink toàn bộ model, không sao chép các file weight lớn.
for src in source.iterdir():
    if src.name in {"config.json", "preprocessor_config.json"}:
        continue

    dst = target / src.name
    if not os.path.lexists(dst):
        dst.symlink_to(src, target_is_directory=src.is_dir())

# Sửa config.json.
with (source / "config.json").open(encoding="utf-8") as f:
    config = json.load(f)

config["model_type"] = "qwen2_5_vl"
config.setdefault(
    "architectures",
    ["Qwen2_5_VLForConditionalGeneration"],
)

with (target / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# Sửa preprocessor_config.json.
with (source / "preprocessor_config.json").open(encoding="utf-8") as f:
    processor_config = json.load(f)

processor_config["image_processor_type"] = "Qwen2VLImageProcessor"
processor_config["processor_class"] = "Qwen2_5_VLProcessor"

with (target / "preprocessor_config.json").open("w", encoding="utf-8") as f:
    json.dump(processor_config, f, ensure_ascii=False, indent=2)

print("Fixed model:", target)
print("model_type:", config["model_type"])
print("image_processor_type:", processor_config["image_processor_type"])

In [1]:
import transformers
print(transformers.__version__)

from transformers import AutoProcessor
print("AutoProcessor import OK")

4.57.1
AutoProcessor import OK


In [2]:
%%bash
set -euo pipefail

export PYTHONPATH=/kaggle/working/transformers_4571_clean:${PYTHONPATH:-}

python - <<'PY'
import transformers
from transformers import AutoProcessor

print("transformers:", transformers.__version__)
print("loaded from:", transformers.__file__)

model_path = "/kaggle/working/qwen2_5_vl_3b_fixed"
processor = AutoProcessor.from_pretrained(
    model_path,
    trust_remote_code=True,
    local_files_only=True,
)
print("processor:", type(processor))
PY

transformers: 4.57.1
loaded from: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
processor: <class 'transformers.models.qwen2_5_vl.processing_qwen2_5_vl.Qwen2_5_VLProcessor'>


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [5]:
%%bash
set -euo pipefail

VIDEO_DATASET=/kaggle/input/datasets/vucongaaa/vce-original-videos
SAMPLE_VIDEO="$(find "$VIDEO_DATASET" -type f -name '06720.mp4' -print -quit)"

test -n "$SAMPLE_VIDEO" || {
  echo "Không tìm thấy 06720.mp4 trong Dataset"
  exit 1
}

VIDEO_ROOT="$(dirname "$SAMPLE_VIDEO")"
echo "VIDEO_ROOT=$VIDEO_ROOT"

python /kaggle/working/DAR/ms-swift/examples/train/sft/dar/prepare_dar_sft_data.py \
  --input /kaggle/input/datasets/vucongaaa/dar-annotation/train.jsonl \
  --output /kaggle/working/dar_sft/train_qwen25vl_ms_sft.jsonl \
  --video-root "$VIDEO_ROOT" \
  --check-videos \
  --strict

VIDEO_ROOT=/kaggle/input/datasets/vucongaaa/vce-original-videos/videos
Wrote 13646 SFT examples to /kaggle/working/dar_sft/train_qwen25vl_ms_sft.jsonl (skipped 0).


In [7]:
%%bash
set -euo pipefail

cd /kaggle/working/DAR/ms-swift/examples/train/sft/dar

export CUDA_VISIBLE_DEVICES=0
export NPROC_PER_NODE=1
export SWIFT_BIN=swift
export GRADIENT_ACCUMULATION_STEPS=32

export MODEL_NAME=/kaggle/working/qwen2_5_vl_3b_fixed
export DATA_JSONL=/kaggle/working/dar_sft/train_qwen25vl_ms_sft.jsonl
export OUTPUT_DIR=/kaggle/working/dar_sft/checkpoints
export PREPARE_DATA=0

bash train_qwen2.5vl_sft_dar.sh \
  --model_type qwen2_5_vl 

Process is terminated.
